# 01 — Project Overview

This notebook documents the project scope, study area, objectives, and model design for the **Oyo Rural Suitability Engine**.

In [1]:
import sys, yaml
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

# Load all configs
configs = {}
for f in (ROOT / "config").glob("*.yml"):
    with open(f) as fh:
        configs[f.stem] = yaml.safe_load(fh)

print("Configs loaded:", list(configs.keys()))


Configs loaded: ['agriculture_config', 'healthcare_config', 'study_area_config']


## Study Area

In [ ]:
sa = configs["study_area_config"]
lgas = sa["study_area"]["lgas"]
print(f"State          : {sa['study_area']['state']}")
print(f"Total LGAs     : {len(lgas)}")
print(f"Projected CRS  : {sa['study_area']['crs_projected']}")
print(f"Resolution (m) : {sa['study_area']['raster_resolution']}")
print(f"Bbox           : W={sa['study_area']['bbox']['west']} E={sa['study_area']['bbox']['east']} S={sa['study_area']['bbox']['south']} N={sa['study_area']['bbox']['north']}")
print()

# Group by senatorial district
from collections import defaultdict
by_dist = defaultdict(list)
for l in lgas:
    by_dist[l["senatorial_district"]].append(l["name"])
for dist, names in sorted(by_dist.items()):
    print(f"{dist} ({len(names)}): {', '.join(names)}")


## Healthcare Model — Criteria & Weights

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

hc = configs["healthcare_config"]["criteria"]
ag = configs["agriculture_config"]["criteria"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, cfg, title, color in [
    (axes[0], hc, "Healthcare Model", "#1D9E75"),
    (axes[1], ag, "Agriculture Model", "#378ADD"),
]:
    names  = [k.replace("_", " ").replace("distance to", "dist.") for k in cfg]
    weights = [v["weight"] for v in cfg.values()]
    bars = ax.barh(names, weights, color=color, alpha=0.85)
    ax.bar_label(bars, labels=[f"{w*100:.1f}%" for w in weights],
                 padding=4, fontsize=9)
    ax.set_xlim(0, max(weights) * 1.35)
    ax.set_title(title, fontsize=12, fontweight="bold", pad=10)
    ax.set_xlabel("AHP Weight", fontsize=9)
    ax.tick_params(labelsize=9)
    ax.spines[["top","right"]].set_visible(False)

plt.suptitle("AHP-Derived Criteria Weights — Both Models", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(ROOT / "outputs/maps/ahp_weights_overview.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: outputs/maps/ahp_weights_overview.png")


## Workflow Summary

```
Raw data → Preprocessing → Criteria derivation → Fuzzy standardisation
→ AHP weighting → WLC + constraints → Classification
→ Uncertainty analysis → Site ranking → Reports + Dashboard
```

See `docs/methodology.md` for full documentation.

In [ ]:
# Print full methodology summary
print("Healthcare model — fuzzy parameter summary")
print("-" * 55)
for name, cfg in hc.items():
    params = cfg["fuzzy_params"]
    param_str = " | ".join(f"{k}={v}" for k,v in params.items())
    print(f"  {name:<35} {cfg['fuzzy_type']:<20} {param_str}")

print()
print("Agriculture model — fuzzy parameter summary")
print("-" * 55)
for name, cfg in ag.items():
    params = cfg["fuzzy_params"]
    param_str = " | ".join(f"{k}={v}" for k,v in params.items())
    print(f"  {name:<35} {cfg['fuzzy_type']:<20} {param_str}")
